In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
datetimes = [(20230228, "28-Feb-2021 10:00:00.123"),
                     (20230220, "20-Feb-2022 08:08:08.999"),
                     (20231031, "31-Dec-2023 11:59:59.123"),
                     (20231130, "31-Aug-2023 00:00:00.000")
                ]

In [ ]:
df_datetimes = spark.createDataFrame(datetimes, schema="date BIGINT, time STRING")

In [ ]:
df_datetimes.show(truncate=False)

In [ ]:
#Преобразуем string в date и timestamp стандартного формата

from pyspark.sql.functions import col, to_date, to_timestamp

df_datetimes. \
    withColumn('to_date', to_date(col('date').cast('string'), 'yyyyMMdd')). \
    withColumn('to_timestamp', to_timestamp(col('time'), 'dd-MMM-yyyy HH:mm:ss.SSS')). \
    show(truncate=False)

In [ ]:
#Далее дополнительные примеры преобразований нестандартного формата в стандартный
from pyspark.sql.functions import lit

df_datetimes.select(lit('02-Mar-2023'), to_timestamp(lit('02-Mar-2023'), 'dd-MMM-yyyy').alias('to_date')).show()

In [ ]:
df_datetimes.select(lit('March 2, 2023'), to_date(lit('March 2, 2023'), 'MMMM d, yyyy').alias('to_date')).show()

In [ ]:
df_datetimes.select(lit('02-March-2023'), to_date(lit('02-March-2023'), 'dd-MMMM-yyyy').alias('to_date')).show()

In [ ]:
df_datetimes.select(lit('02/03/2023'), to_date(lit('02/03/2023'), 'dd/MM/yyyy').alias('to_date')).show()

In [ ]:
df_datetimes.select(lit('20230302'), to_date(lit('20230302'), 'yyyyMMdd').alias('to_date')).show()

In [ ]:
#Пример преобразования с помощью date_format
#Обратите внимание как сначала мы преобразуем BIGINT в String, далее применяем to_data и преобразуем с помощью date_format

from pyspark.sql.functions import date_format

df_datetimes. \
    withColumn("date_desc", date_format(to_date(col('date').cast('string'), 'yyyyMMdd'), "MMMM d, yyyy")). \
    show(truncate=False)

In [ ]:
# Получаем день недели с помощью date_format
df_datetimes. \
    withColumn("day_name_abbr", date_format(to_date(col('date').cast('string'), 'yyyyMMdd'), "EE")). \
    show(truncate=False)

In [ ]:
# Получаем день недели с помощью date_format
df_datetimes. \
    withColumn("day_name_full", date_format(to_date(col('date').cast('string'), 'yyyyMMdd'), "EEEE")). \
    show(truncate=False)

In [ ]:
unixtimes = [(1393561800, ),
             (1456713488, ),
             (1514701799, ),
             (1567189800, )
            ]

In [ ]:
df_unixtimes = spark.createDataFrame(unixtimes).toDF("unixtime")

df_unixtimes.show()

In [ ]:
df_unixtimes.printSchema()

In [ ]:
#Часто приходится работать с unixtime. Для решения этой задачи пригодится from_unixtime

from pyspark.sql.functions import from_unixtime

df_unixtimes. \
    withColumn("date", from_unixtime("unixtime", "yyyy-MM-dd")). \
    withColumn("time", from_unixtime("unixtime")). \
    show()

help(from_unixtime)

In [ ]:
spark.stop()